# Lab 16 — Reinforcement Learning with Verifiable Rewards
**Goal:** Build a tiny environment where a policy receives a deterministic reward for correct arithmetic answers.

Free-first: CPU or free Colab where possible; expensive paths are clearly marked optional.



In [ ]:
import random
random.seed(0)
problems=[(1,2,3),(2,7,9),(5,5,10),(9,4,13),(6,8,14)]
print(problems)


In [ ]:
def reward(pred,target):
    try:return 1.0 if int(pred)==target else 0.0
    except:return 0.0
preds=[str(random.randint(0,15)) for _ in problems]
print([(p,pred,reward(pred,target)) for p,pred in zip(problems,preds) for target in [p[2]]])


In [ ]:
# Compare random behavior with a perfect verifier-compatible policy.
random_score=sum(reward(str(random.randint(0,15)),t) for _,_,t in problems)/len(problems)
print("random sample score:",random_score)
print("perfect score:",1.0)


### Critical thinking
A reward function is an executable definition of success. If the verifier is incomplete, agents can optimize the proxy rather than the intended objective. That is reward hacking.


## Primary source
https://cs336.stanford.edu/


## Deepening: verifier quality controls the reward

This section turns the notebook into a quantitative lab. Predict before executing.


In [ ]:
def exact_reward(answer, target):
    return 1.0 if answer.strip() == target else 0.0

def heuristic_reward(answer):
    # Deliberately flawed proxy: rewards long answers.
    return min(len(answer.split()) / 20.0, 1.0)

answers = ["42", "The correct answer is 42 because the calculation gives forty two."]
for a in answers:
    print(a, "exact=", exact_reward(a,"42"), "heuristic=", heuristic_reward(a))


## Break experiment

Optimize against the heuristic reward mentally.

Which answer would a policy prefer?

Then compare that result with the true objective.

### Report

objective → verifier → proxy → exploit → protected metric → repair

Complete templates/EXPERIMENT_CARD.md for the final result.


## 7. Research exercise — real reasoning records and verifier signals


In [ ]:
from datasets import load_dataset
from collections import Counter

ds=load_dataset("open-r1/OpenR1-Math-220k",split="train",streaming=True)
rows=[]
for row in ds:
    if row.get("problem") and (row.get("solution") or row.get("answer") is not None):
        rows.append(row)
    if len(rows)>=300: break

print("rows:",len(rows))
print("fields:",list(rows[0].keys()) if rows else [])
for r in rows[:3]:
    print("\nPROBLEM:",str(r.get("problem"))[:300])
    print("ANSWER:",str(r.get("answer"))[:120])
    print("SOURCE:",r.get("source"))


<details><summary><strong>Expected findings / answer</strong></summary>

A verifiable-reward setup needs a checkable target, not merely a plausible-looking completion. Math data is useful because final answers can often be validated independently of the reasoning trace. The key distinction is **reward signal quality**: a verifier can score correctness of the target answer without proving that every intermediate explanation is valid. This motivates separate process and outcome checks.
</details>
